# Point Cloud Processing Pipeline
This notebook performs three processing steps on point cloud data stored as `.txt` files:
1. Distance-based downsampling
2. Tiling in GPS-time direction (120 m tiles)
3. Local coordinate shifting using tile center

In [ ]:
# ── Cell 1 ─────────────────────────────────────────────────────────────────
# Memory-efficient distance-based downsampling using chunked processing.
# First 3 columns = X, Y, Z. All other columns preserved as-is.
# ──────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
from pathlib import Path

# ── Parameters ────────────────────────────────────────────────────────────
INPUT_TXT  = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch/Dataset/Raw/PC.txt")
OUTPUT_TXT = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch/Dataset/Preparation/0.Downsampled/PC_DS5cm.txt")
MIN_DIST   = 0.05       # minimum distance between kept points (metres)
SEP        = r"\s+"     # handles any number of spaces or tabs
CHUNK_SIZE = 500_000    # rows per chunk — lower if still crashing (e.g. 100_000)
# ──────────────────────────────────────────────────────────────────────────

X_COL, Y_COL, Z_COL = 0, 1, 2

seen_voxels = set()   # global voxel registry across all chunks
total_in    = 0
total_out   = 0
first_chunk = True

with pd.read_csv(INPUT_TXT, sep=SEP, header=None, engine="python",
                 chunksize=CHUNK_SIZE) as reader:

    with open(OUTPUT_TXT, "w") as out_f:

        for chunk in reader:
            total_in += len(chunk)

            # ── Voxel keys for this chunk ──────────────────────────────
            coords     = chunk[[X_COL, Y_COL, Z_COL]].values
            voxel_keys = np.floor(coords / MIN_DIST).astype(np.int64)

            # ── Filter: keep only points in unseen voxels ──────────────
            keep_rows = []
            for i, vk in enumerate(voxel_keys):
                key = (vk[0], vk[1], vk[2])   # tuple is hashable & fast
                if key not in seen_voxels:
                    seen_voxels.add(key)
                    keep_rows.append(i)

            if not keep_rows:
                continue

            filtered = chunk.iloc[keep_rows]
            total_out += len(filtered)

            # ── Write chunk (no header, space-separated) ───────────────
            filtered.to_csv(out_f, sep=" ", index=False, header=False)

            print(f"  processed {total_in:>10,} pts | kept so far: {total_out:,} "
                  f"({100*total_out/total_in:.1f}%)  voxels: {len(seen_voxels):,}",
                  end="\r")

print(f"\nDone. {total_in:,} → {total_out:,} points "
      f"({100*total_out/total_in:.1f}% retained)")
print(f"Saved → '{OUTPUT_TXT}'")

  processed 95,057,933 pts | kept so far: 35,829,687 (37.7%)  voxels: 35,829,687
Done. 95,057,933 → 35,829,687 points (37.7% retained)
Saved → '/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch-master/PC_DS5cm.txt'


In [ ]:
# ── Cell 2 ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path

# ── Parameters ────────────────────────────────────────────────────────────
INPUT_TXT   = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch/Dataset/Preparation/0.Downsampled/PC_DS5cm.txt")
OUTPUT_DIR  = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch/Dataset/Preparation/1.Tiled")
TILE_LENGTH = 120.0
OVERLAP     = 0.0
SEP         = r"\s+"
CHUNK_SIZE  = 500_000
GPS_COL     = -4          # 4th column from end
X_COL, Y_COL = 0, 1
# ──────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Pass 1: read X, Y, GPS ─────────────────────────────────────────────────
print("=" * 60)
print("PASS 1: Reading X, Y, GPSTime ...")
print("=" * 60)

chunks_nav = []
n_cols     = None

for i, chunk in enumerate(pd.read_csv(INPUT_TXT, sep=SEP, header=None,
                                       engine="python", chunksize=CHUNK_SIZE)):
    if n_cols is None:
        n_cols  = chunk.shape[1]
        gps_idx = n_cols + GPS_COL if GPS_COL < 0 else GPS_COL
        print(f"  Total columns  : {n_cols}")
        print(f"  GPSTime column : {gps_idx}")
    chunks_nav.append(chunk[[X_COL, Y_COL, gps_idx]].astype(np.float64))
    print(f"  Chunk {i+1:>3} — {(i+1)*CHUNK_SIZE:,} rows so far ...")

nav = pd.concat(chunks_nav, ignore_index=True)
nav.columns = ["X", "Y", "GPS"]
print(f"\n  Total points : {len(nav):,}")

# ── PCA to find principal travel direction ─────────────────────────────────
print("\nFinding principal travel direction via PCA ...")

# Sample up to 500k points for PCA (no need for all 35M)
sample = nav[["X", "Y"]].sample(min(500_000, len(nav)), random_state=0).values
centroid = sample.mean(axis=0)
sample_c = sample - centroid

cov      = np.cov(sample_c.T)
eigvals, eigvecs = np.linalg.eigh(cov)
main_axis = eigvecs[:, np.argmax(eigvals)]   # unit vector of travel direction

# Make sure it points in positive GPS-time direction
# (sort a small sample by GPS and check dot product)
gps_sample = nav.sample(min(10_000, len(nav)), random_state=0).sort_values("GPS")
xy_start   = gps_sample.iloc[:100][["X", "Y"]].values.mean(axis=0)
xy_end     = gps_sample.iloc[-100:][["X", "Y"]].values.mean(axis=0)
if np.dot(xy_end - xy_start, main_axis) < 0:
    main_axis = -main_axis

angle_deg = np.degrees(np.arctan2(main_axis[1], main_axis[0]))
print(f"  Principal axis angle : {angle_deg:.1f}° from East")
print(f"  Direction vector     : ({main_axis[0]:.4f}, {main_axis[1]:.4f})")

# ── Project all points onto principal axis ────────────────────────────────
print("\nProjecting all points onto principal axis ...")
xy_centered  = nav[["X", "Y"]].values - centroid
projections  = xy_centered @ main_axis          # scalar projection per point
nav["Proj"]  = projections

proj_min = projections.min()
proj_max = projections.max()
total_length = proj_max - proj_min
print(f"  Projection range : {proj_min:.2f} → {proj_max:.2f} m")
print(f"  Total length     : {total_length:.2f} m")

# ── Tile assignment ────────────────────────────────────────────────────────
print("\nAssigning tile indices ...")
step         = TILE_LENGTH - OVERLAP
shifted_proj = projections - proj_min
tile_indices = np.floor(shifted_proj / step).astype(np.int32)
nav["tile"]  = tile_indices

n_tiles = tile_indices.max() + 1
print(f"  Number of tiles : {n_tiles}  "
      f"({total_length:.1f} m / {TILE_LENGTH} m)")

# orig_idx → tile lookup
nav = nav.reset_index(drop=False)
nav.rename(columns={"index": "orig_idx"}, inplace=True)
orig_to_tile = dict(zip(nav["orig_idx"].astype(int), nav["tile"].astype(int)))

# ── Pass 2: stream full file, write tiles ─────────────────────────────────
print("\n" + "=" * 60)
print("PASS 2: Streaming full file and writing tiles ...")
print("=" * 60)

tile_files  = {}
tile_counts = {}
row_global  = 0

for i, chunk in enumerate(pd.read_csv(INPUT_TXT, sep=SEP, header=None,
                                       engine="python", chunksize=CHUNK_SIZE)):
    written = 0
    for local_i in range(len(chunk)):
        orig_idx = row_global + local_i
        if orig_idx not in orig_to_tile:
            continue
        t_idx   = orig_to_tile[orig_idx]
        row_str = " ".join(chunk.iloc[local_i].astype(str).tolist()) + "\n"
        if t_idx not in tile_files:
            tile_files[t_idx]  = open(OUTPUT_DIR / f"tile_{t_idx:04d}.txt", "w")
            tile_counts[t_idx] = 0
        tile_files[t_idx].write(row_str)
        tile_counts[t_idx] += 1
        written += 1

    row_global += len(chunk)
    print(f"  Chunk {i+1:>3} — rows: {row_global:,}  written: {written:,}  tiles open: {len(tile_files)}")

for fh in tile_files.values():
    fh.close()

# ── Summary ───────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("TILING COMPLETE — Summary:")
print("=" * 60)
for t_idx in sorted(tile_counts):
    t_start = proj_min + t_idx * step
    t_end   = t_start + TILE_LENGTH
    print(f"  tile_{t_idx:04d}.txt : {tile_counts[t_idx]:>8,} pts  "
          f"[{t_start:.1f} – {t_end:.1f} m along principal axis]")
print(f"\n  Principal axis angle : {angle_deg:.1f}° from East")
print(f"  Total tiles saved    : {len(tile_files)}")
print(f"  Output folder        : '{OUTPUT_DIR}'")
print("=" * 60)

PASS 1: Reading X, Y, GPSTime ...
  Total columns  : 12
  GPSTime column : 8
  Chunk   1 — 500,000 rows so far ...
  Chunk   2 — 1,000,000 rows so far ...
  Chunk   3 — 1,500,000 rows so far ...
  Chunk   4 — 2,000,000 rows so far ...
  Chunk   5 — 2,500,000 rows so far ...
  Chunk   6 — 3,000,000 rows so far ...
  Chunk   7 — 3,500,000 rows so far ...
  Chunk   8 — 4,000,000 rows so far ...
  Chunk   9 — 4,500,000 rows so far ...
  Chunk  10 — 5,000,000 rows so far ...
  Chunk  11 — 5,500,000 rows so far ...
  Chunk  12 — 6,000,000 rows so far ...
  Chunk  13 — 6,500,000 rows so far ...
  Chunk  14 — 7,000,000 rows so far ...
  Chunk  15 — 7,500,000 rows so far ...
  Chunk  16 — 8,000,000 rows so far ...
  Chunk  17 — 8,500,000 rows so far ...
  Chunk  18 — 9,000,000 rows so far ...
  Chunk  19 — 9,500,000 rows so far ...
  Chunk  20 — 10,000,000 rows so far ...
  Chunk  21 — 10,500,000 rows so far ...
  Chunk  22 — 11,000,000 rows so far ...
  Chunk  23 — 11,500,000 rows so far ...
 

In [ ]:
# ── Cell 3 ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path

# ── Parameters ────────────────────────────────────────────────────────────
TILES_DIR       = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch/Dataset/Preparation/1.Tiled")
SEP             = r"\s+"      # handles any number of spaces or tabs
X_COL, Y_COL, Z_COL = 0, 1, 2  # first 3 columns are always X, Y, Z
USE_BBOX_CENTER = True           # True = bounding-box centre, False = centroid
# ──────────────────────────────────────────────────────────────────────────

tile_files = sorted(TILES_DIR.glob("*.txt"))
tile_files = [f for f in tile_files if not f.stem.endswith("_local")]
print(f"Found {len(tile_files)} tile file(s) in '{TILES_DIR}'")

origin_records = []

for tile_path in tile_files:
    df = pd.read_csv(tile_path, sep=SEP, header=None, engine="python")

    if df.empty:
        print(f"  [SKIP] '{tile_path.name}' is empty.")
        continue

    # ── Compute centre from columns 0, 1, 2 (X, Y, Z) ────────────────
    if USE_BBOX_CENTER:
        X_c = (df[X_COL].max() + df[X_COL].min()) / 2.0
        Y_c = (df[Y_COL].max() + df[Y_COL].min()) / 2.0
        Z_c = (df[Z_COL].max() + df[Z_COL].min()) / 2.0
        method = "bbox_center"
    else:
        X_c = df[X_COL].mean()
        Y_c = df[Y_COL].mean()
        Z_c = df[Z_COL].mean()
        method = "centroid"

    # ── Shift X, Y, Z — all other columns untouched ───────────────────
    df_local = df.copy()
    df_local[X_COL] = df[X_COL] - X_c
    df_local[Y_COL] = df[Y_COL] - Y_c
    df_local[Z_COL] = df[Z_COL] - Z_c

    # ── Save local tile (same folder) ─────────────────────────────────
    out_path = tile_path.parent / (tile_path.stem + "_local.txt")
    df_local.to_csv(out_path, sep=" ", index=False, header=False)

    # ── Save origin metadata ───────────────────────────────────────────
    origin_df = pd.DataFrame([{"tile":     tile_path.name,
                                "method":   method,
                                "X_origin": X_c,
                                "Y_origin": Y_c,
                                "Z_origin": Z_c,
                                "n_cols":   df.shape[1],
                                "n_pts":    len(df)}])
    origin_df.to_csv(tile_path.parent / (tile_path.stem + "_local_origin.csv"),
                     index=False)
    origin_records.append(origin_df)

    print(f"  '{tile_path.name}' ({df.shape[1]} cols, {len(df):,} pts) → "
          f"'{out_path.name}'  "
          f"origin=({X_c:.3f}, {Y_c:.3f}, {Z_c:.3f})  [{method}]")

# ── Summary CSV ───────────────────────────────────────────────────────────
if origin_records:
    summary_path = TILES_DIR / "all_tiles_origins.csv"
    pd.concat(origin_records, ignore_index=True).to_csv(summary_path, index=False)
    print(f"\nOrigin summary saved → '{summary_path}'")

print("\nLocal-coordinate conversion complete.")

Found 15 tile file(s) in '/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch-master/tiles'
  'tile_0000.txt' (12 cols, 1,870,702 pts) → 'tile_0000_local.txt'  origin=(581489.296, 4413320.262, 206.599)  [bbox_center]
  'tile_0001.txt' (12 cols, 2,617,717 pts) → 'tile_0001_local.txt'  origin=(581484.764, 4413200.839, 208.309)  [bbox_center]
  'tile_0002.txt' (12 cols, 2,775,140 pts) → 'tile_0002_local.txt'  origin=(581488.132, 4413081.766, 213.351)  [bbox_center]
  'tile_0003.txt' (12 cols, 3,564,065 pts) → 'tile_0003_local.txt'  origin=(581496.820, 4412961.525, 216.579)  [bbox_center]
  'tile_0004.txt' (12 cols, 2,807,943 pts) → 'tile_0004_local.txt'  origin=(581503.505, 4412841.488, 216.432)  [bbox_center]
  'tile_0005.txt' (12 cols, 2,544,819 pts) → 'tile_0005_local.txt'  origin=(581492.693, 4412721.639, 219.775)  [bbox_center]
  'tile_0006.txt' (12 cols, 2,973,477 pts) → 'tile_0006_local.txt'  origin=(581500.749, 4412601.734, 223.061)  [bbox_center]
  'tile_0007.txt' (12 

In [ ]:
# ── Cell 4 ─────────────────────────────────────────────────────────────────
# Read local-coordinate tiles, keep only selected columns in specified order,
# and save.
#
# Original file columns (0-indexed):
#   0: X  |  1: Y  |  2: Z  |  3: Intensity  |  7: PointSourceID  |  8: GPSTime
#
# Output column order: X, Y, Z, GPSTime, Intensity, PointSourceID
# Output: <tile>_local_selected.txt  (same folder)
# ──────────────────────────────────────────────────────────────────────────

import pandas as pd
from pathlib import Path

# ── Parameters ────────────────────────────────────────────────────────────
TILES_DIR = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch/Dataset/Preparation/2.Localized")
SEP       = r"\s+"

#        X  Y  Z   Intensity  PointSourceID GPSTime 
KEEP_COLS = [0, 1, 2, 3, 7, 8]
# ──────────────────────────────────────────────────────────────────────────

tile_files = sorted(TILES_DIR.glob("*_local.txt"))
print(f"Found {len(tile_files)} local tile file(s) in '{TILES_DIR}'")

for tile_path in tile_files:
    df = pd.read_csv(tile_path, sep=SEP, header=None, engine="python")

    if df.empty:
        print(f"  [SKIP] '{tile_path.name}' is empty.")
        continue

    missing = [c for c in KEEP_COLS if c >= df.shape[1]]
    if missing:
        print(f"  [SKIP] '{tile_path.name}' only has {df.shape[1]} cols — "
              f"missing indices {missing}")
        continue

    df_selected = df[KEEP_COLS]

    out_path = tile_path.parent / (tile_path.stem + "_selected.txt")
    df_selected.to_csv(out_path, sep="\t", index=False, header=False)

    print(f"  '{tile_path.name}'  {df.shape[1]} cols × {len(df):,} pts  "
          f"→  '{out_path.name}'  (6 cols: X Y Z GPSTime Intensity PointSourceID)")

print("\nColumn selection complete.")

Found 15 local tile file(s) in '/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch-master/tiles/local'
  'tile_0000_local.txt'  12 cols × 1,870,702 pts  →  'tile_0000_local_selected.txt'  (6 cols: X Y Z GPSTime Intensity PointSourceID)
  'tile_0001_local.txt'  12 cols × 2,617,717 pts  →  'tile_0001_local_selected.txt'  (6 cols: X Y Z GPSTime Intensity PointSourceID)
  'tile_0002_local.txt'  12 cols × 2,775,140 pts  →  'tile_0002_local_selected.txt'  (6 cols: X Y Z GPSTime Intensity PointSourceID)
  'tile_0003_local.txt'  12 cols × 3,564,065 pts  →  'tile_0003_local_selected.txt'  (6 cols: X Y Z GPSTime Intensity PointSourceID)
  'tile_0004_local.txt'  12 cols × 2,807,943 pts  →  'tile_0004_local_selected.txt'  (6 cols: X Y Z GPSTime Intensity PointSourceID)
  'tile_0005_local.txt'  12 cols × 2,544,819 pts  →  'tile_0005_local_selected.txt'  (6 cols: X Y Z GPSTime Intensity PointSourceID)
  'tile_0006_local.txt'  12 cols × 2,973,477 pts  →  'tile_0006_local_selected.txt'  (6